In [ ]:
from __future__ import annotationsfrom pathlib import Pathfrom typing import List, Tuple, Dict, Any, Set, Optionalimport math, colorsys, itertools, random, re, sys, time, osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlib.animation as animationfrom matplotlib.colors import ListedColormap, to_rgb, to_hexfrom matplotlib.patches import Patch, Rectanglefrom mpl_toolkits.mplot3d import Axes3Dfrom IPython.display import Video, displayBASE_COLOURS = {    "NewMC": "#ffcc99", "DR": "#ffcccc", "NewMC_Expand": "#ccffcc",    "NewMC_Restricted": "#ccccff", "NewMC_Biased": "#ffffcc",    "NewMC_Restricted_Expand": "#ffccff", "NewMC_Biased_Expand": "#ccffff",    "Kawasaki": "#ffcccc", "Kawasaki_Black": "#000000",}def _anchor_colour(stem: str) -> str:    for k in sorted(BASE_COLOURS, key=len, reverse=True):        if stem.startswith(k): return BASE_COLOURS[k]    return "#808080"def _tint_red(hex_colour: str, factor: float) -> str:    r, g, b = to_rgb(hex_colour)    r = min(1.0, r * factor)    return to_hex((r, g, b))def _tint_green(hex_colour: str, factor: float) -> str:    r, g, b = to_rgb(hex_colour)    g = min(1.0, g * factor)    return to_hex((r, g, b))def get_distinct_colors(n):    colors = ["#000000"]    if n > 1:        for i in range(n - 1):            hue = (i / (n - 1)) * 0.8 + 0.1            saturation = 0.8            value = 0.75            rgb = colorsys.hsv_to_rgb(hue, saturation, value)            colors.append(to_hex(rgb))    return colorsdef build_filenames(*, BJ: str, include_kawasaki: bool,                   base_variants: List[str], signifiers: List[List[str]], cluster_types: List[str]                   ) -> Tuple[List[str], List[str], List[str], List[str], List[str], List[str]]:    csv, tbl, col, leg, leg_col, expand = [], [], [], [], [], []        if include_kawasaki:        fn = f"BJ={BJ}_Kawasaki_0_tol0.00_scp0.00.csv"        csv.append(fn)        tbl.append("Kawasaki")        col.append(BASE_COLOURS["Kawasaki_Black"])        leg.append("Kawasaki")        leg_col.append(BASE_COLOURS["Kawasaki_Black"])        for stem in base_variants:        for ct in cluster_types:            anchor = _anchor_colour(stem)            name = f"{stem}_{ct}"            for j, signifier_1 in enumerate(signifiers[0]):                for k, signifier_2 in enumerate(signifiers[1]):                    fn_signifier = f"BJ={BJ}_{name}_{signifier_1}_{signifier_2}.csv"                    csv.append(fn_signifier)                    tbl.append(f"{name}_{signifier_1}_{signifier_2}")                                        tint1 = 0.8 + (0.4 * j / len(signifiers[0]))                    tint2 = 0.8 + (0.4 * k / len(signifiers[1]))                    tinted_color = _tint_red(_tint_green(anchor, tint1), tint2)                    col.append(tinted_color)                                        if k == 0 or k == len(signifiers[1]) - 1:                        leg.append(f"{name}_{signifier_1}_{signifier_2}")                        leg_col.append(tinted_color)                    expand.append(fn_signifier)    return csv, tbl, col, leg, leg_col, expanddef parse_grid_config_string_to_species_array(config_raw: any, N: int) -> Optional[np.ndarray]:    if pd.isna(config_raw) or str(config_raw).strip() in ["", '""', "''"]:        return None    s = str(config_raw).strip('"\' ')    if not s:        return None    parts = s.split(';')    if len(parts) != N * N:        return None    species = np.zeros((N, N), dtype=np.int8)    try:        for i, cell_data_str in enumerate(parts):            if not cell_data_str.strip():                 continue            cell_parts = cell_data_str.split('|')            if len(cell_parts) == 0:                continue            species_val = int(cell_parts[0])            y, x = i // N, i % N            species[y, x] = species_val    except (ValueError, IndexError):        return None    return species

In [ ]:
def energy_analysis_driver(csv_files, table_labels, curve_colours, title="Energy Analysis"):    plt.rcParams.update({'font.size': 13})    fig1 = plt.figure(figsize=(20, 15))    gs1 = fig1.add_gridspec(3, 3, height_ratios=[1, 1, 1], hspace=0.4)        ax_raw = fig1.add_subplot(gs1[0, :])    ax_E, ax_dE, ax_simple = fig1.add_subplot(gs1[1, 0]), fig1.add_subplot(gs1[1, 1]), fig1.add_subplot(gs1[1, 2])    ax_slope = fig1.add_subplot(gs1[2, :])        fig1.suptitle(title + " - Energy Plots", fontsize=22, fontweight="bold", y=0.95)        lengths = [len(pd.read_csv(f)) for f in csv_files if Path(f).is_file()]    if not lengths: return    W = max(50, min(1000, min(lengths) // 10))    thr = 1e-4    streak = max(3, min(10, W // 100))    max_steps = 0        T_data = {k: [] for k in ("run", "Efin", "conv_iter", "conv_E", "norm_Efin", "norm_conv_iter", "norm_conv_E", "acceptance_rate")}        for fp, lbl_tbl, col in zip(csv_files, table_labels, curve_colours):        p = Path(fp)        if not p.is_file(): continue        try:            df = pd.read_csv(p)            if "Energy" not in df.columns: continue                        steps = np.arange(len(df))            max_steps = max(max_steps, len(df))            E = df["Energy"].ffill().bfill()            ax_raw.plot(steps, E, lw=1, color=col, alpha=0.5, label=lbl_tbl)                        cum = E.expanding().mean()            ax_E.plot(steps, cum, lw=2, color=col, alpha=0.7)            ax_simple.plot(steps, E, lw=1, color=col, alpha=0.7, label=lbl_tbl)                        if "Total_Acceptances" in df.columns:                acceptance_rate = (df["Total_Acceptances"].iloc[-1] * 100.0 / len(df)) if len(df) > 0 else 0.0            else:                acceptance_rate = float('nan')                        wn = None            if "Norm" in df.columns:                N_norm = df["Norm"].replace([np.inf, -np.inf, 0], np.nan).ffill().bfill()                eff = (1 - N_norm) * 0 / N_norm + 1                wn = (E * eff).cumsum() / eff.cumsum()                ax_E.plot(steps, wn, lw=2, ls="--", color=col, alpha=0.7)                        stride = max(1, len(steps) // 1000)            ax_dE.scatter(steps[1::stride], np.abs(np.diff(cum))[::stride], s=10, alpha=.7, color=col)                        conv = nconv = None            if len(cum) >= 2 * W:                idx = np.arange(0, len(cum) - W, W)                sl = np.array([(cum[i + W] - cum[i]) / W for i in idx])                ax_slope.scatter(idx + W / 2, np.abs(sl), s=24, alpha=.8, color=col)                conv = next((int(idx[i]) for i in range(len(sl) - streak + 1)                           if np.all(np.abs(sl[i:i + streak]) < thr)), None)                        T_data["run"].append(lbl_tbl)            T_data["Efin"].append(f"{cum.iloc[-1]:.4g}")            T_data["conv_iter"].append(conv if conv is not None else "¬conv")            T_data["conv_E"].append(f"{cum[conv]:.4g}" if conv is not None else "N/A")            T_data["acceptance_rate"].append(f"{acceptance_rate:.2f}%" if not np.isnan(acceptance_rate) else "N/A")            if wn is not None:                T_data["norm_Efin"].append(f"{wn.iloc[-1]:.4g}")                T_data["norm_conv_iter"].append(nconv if nconv is not None else "N/A")                T_data["norm_conv_E"].append(f"{wn[nconv]:.4g}" if nconv is not None else "N/A")            else:                T_data["norm_Efin"].append("N/A")                T_data["norm_conv_iter"].append("N/A")                T_data["norm_conv_E"].append("N/A")        except Exception:            continue        ax_raw.set(xlabel="step", ylabel="E"); ax_raw.grid(ls=":")    ax_raw.legend(fontsize=10)    ax_E.set(xscale="log", xlabel="step", ylabel="⟨E⟩"); ax_E.grid(ls=":")    ax_dE.set(xscale="log", yscale="log", xlim=(1, max_steps), xlabel="step", ylabel="|Δ⟨E⟩|"); ax_dE.grid(ls=":")    ax_simple.set(xscale="log", xlim=(1, max_steps), xlabel="step", ylabel="E"); ax_simple.grid(ls=":")    ax_simple.legend(fontsize=8, loc='best')    ax_slope.set(xscale="log", yscale="log", xlabel="step", ylabel=f"|slope| (W={W})"); ax_slope.grid(ls=":")        plt.tight_layout()    plt.show()        num_rows = len(T_data["run"])    if num_rows > 0:        fig2 = plt.figure(figsize=(20, max(8, 2 + num_rows * 0.5)))        ax_tbl = fig2.add_subplot(111)        ax_tbl.axis("off")        fig2.suptitle(title + " - Convergence Table", fontsize=22, fontweight="bold", y=0.95)                cell_text = list(zip(T_data["run"], T_data["Efin"], T_data["conv_iter"], T_data["conv_E"],                           T_data["norm_Efin"], T_data["norm_conv_iter"], T_data["norm_conv_E"],                           T_data["acceptance_rate"]))                col_labels = ["run", "final E", "conv iter", "conv E", "Norm Efin", "Norm conv iter", "Norm conv E", "Accept Rate"]                tbl = ax_tbl.table(cellText=cell_text, colLabels=col_labels, loc="center", cellLoc="center")        font_size = min(12, 160 / num_rows)        tbl.auto_set_font_size(False)        tbl.set_fontsize(font_size)        tbl.scale(min(2.0, 20 / num_rows), min(2.0, 20 / num_rows) * 1.2)                for cell in tbl._cells.values():            cell.set_edgecolor('black')            cell.set_facecolor('white')            cell.set_alpha(0.8)                plt.tight_layout()        plt.show()def cluster_analysis_driver(csv_files, table_labels, curve_colours):    for csv_file, table_label, curve_color in zip(csv_files, table_labels, curve_colours):        file_path = Path(csv_file)        if not file_path.is_file():            continue                try:            df = pd.read_csv(file_path)            boundary_cols = [col for col in df.columns if col.startswith('BoundarySize_Cluster')]            interior_cols = [col for col in df.columns if col.startswith('InteriorSize_Cluster')]                        cluster_numbers = []            for col in boundary_cols:                cluster_num = int(col.replace('BoundarySize_Cluster', ''))                cluster_numbers.append(cluster_num)            cluster_numbers = sorted(list(set(cluster_numbers)))                        if len(cluster_numbers) == 0:                continue                        fig = plt.figure(figsize=(16, 10))            fig.suptitle(f'Cluster Analysis: {table_label}', fontsize=16, fontweight='bold')                        if len(cluster_numbers) <= 2:                rows, cols = 1, len(cluster_numbers)            elif len(cluster_numbers) <= 4:                rows, cols = 2, 2            elif len(cluster_numbers) <= 6:                rows, cols = 2, 3            elif len(cluster_numbers) <= 9:                rows, cols = 3, 3            else:                rows, cols = 4, int(np.ceil(len(cluster_numbers) / 4))                        iterations = np.arange(len(df))            cluster_colors = get_distinct_colors(len(cluster_numbers))                        for i, cluster_num in enumerate(cluster_numbers):                ax = fig.add_subplot(rows, cols, i + 1)                                boundary_col = f'BoundarySize_Cluster{cluster_num}'                interior_col = f'InteriorSize_Cluster{cluster_num}'                                if boundary_col in df.columns and interior_col in df.columns:                    boundary_data = df[boundary_col].fillna(0)                    interior_data = df[interior_col].fillna(0)                                        ax.plot(iterations, boundary_data, label='Boundary', color=cluster_colors[i], linewidth=2, alpha=0.8)                    ax.plot(iterations, interior_data, label='Interior', color=cluster_colors[i], linewidth=2, linestyle='--', alpha=0.8)                                        total_size = boundary_data + interior_data                    ax.plot(iterations, total_size, label='Total', color='gray', linewidth=1, alpha=0.6)                                        ax.set_title(f'Cluster {cluster_num}', fontweight='bold')                    ax.set_xlabel('Iteration')                    ax.set_ylabel('Number of Cells')                    ax.grid(True, alpha=0.3)                    ax.legend(fontsize=8)                                        final_boundary = boundary_data.iloc[-1] if len(boundary_data) > 0 else 0                    final_interior = interior_data.iloc[-1] if len(interior_data) > 0 else 0                    final_total = final_boundary + final_interior                                        stats_text = f'Final: B={int(final_boundary)}, I={int(final_interior)}, T={int(final_total)}'                    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=8, verticalalignment='top',                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))                        plt.tight_layout()            plt.show()        except Exception:            continuedef grid_visualization_driver(csv_files, table_labels):    for csv_file, table_label in zip(csv_files, table_labels):        config_file = csv_file.replace('.csv', '_GridConfig.csv')        config_path = Path(config_file)                if not config_path.is_file():            continue                try:            cfg_df = pd.read_csv(config_path)            if cfg_df.empty:                continue                        required_cols = ["x", "y", "species", "cluster_id", "cell_type"]            if not all(col in cfg_df.columns for col in required_cols):                continue                        N = int(max(cfg_df["x"].max(), cfg_df["y"].max()) + 1) if not cfg_df.empty else 0                        fig, ax = plt.subplots(1, 1, figsize=(12, 10))            ax.set(xlim=(-0.5, N - 0.5), ylim=(-0.5, N - 0.5), aspect="equal", xticks=[], yticks=[])            ax.set_title(f"Grid Configuration: {table_label}", fontweight="bold", fontsize=14)                        unique_clusters = sorted(cfg_df["cluster_id"].unique())                        for _, r in cfg_df.iterrows():                if r["species"] == 0:                    base_color = "#d62728"                elif r["species"] == 1:                    base_color = "#1f77b4"                else:                    base_color = "#808080"                                if r["cell_type"] == "INTERIOR":                    alpha = 0.4                elif r["cell_type"] == "BOUNDARY":                    alpha = 0.8                else:                    alpha = 0.6                                rect = plt.Rectangle((r["x"] - 0.4, r["y"] - 0.4), 0.8, 0.8,                                   facecolor=base_color, alpha=alpha, edgecolor='black', linewidth=0.5)                ax.add_patch(rect)                                ax.text(r["x"], r["y"], str(r["cluster_id"]), ha="center", va="center", fontsize=8, color='white', weight='bold')                        legend_elements = [                Patch(facecolor="#d62728", alpha=0.8, label='Species 0 (Boundary)'),                Patch(facecolor="#d62728", alpha=0.4, label='Species 0 (Interior)'),                Patch(facecolor="#1f77b4", alpha=0.8, label='Species 1 (Boundary)'),                Patch(facecolor="#1f77b4", alpha=0.4, label='Species 1 (Interior)')            ]            ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)                        for i in range(N + 1):                ax.axhline(i - 0.5, color='gray', linewidth=0.3, alpha=0.5)                ax.axvline(i - 0.5, color='gray', linewidth=0.3, alpha=0.5)                        cluster_stats = []            for cluster_id in unique_clusters:                cluster_cells = cfg_df[cfg_df["cluster_id"] == cluster_id]                boundary_count = len(cluster_cells[cluster_cells["cell_type"] == "BOUNDARY"])                interior_count = len(cluster_cells[cluster_cells["cell_type"] == "INTERIOR"])                total_count = len(cluster_cells)                cluster_stats.append(f"Cluster {cluster_id}: B={boundary_count}, I={interior_count}, Total={total_count}")                        stats_text = "\n".join(cluster_stats)            ax.text(1.02, 0.5, stats_text, transform=ax.transAxes, fontsize=9, verticalalignment='center',                   bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))                        plt.tight_layout()            plt.show()        except Exception:            continuedef moves_analysis_driver(csv_files, table_labels, curve_colours):    for csv_file, table_label, curve_color in zip(csv_files, table_labels, curve_colours):        file_path = Path(csv_file)        if not file_path.is_file():            continue                try:            df = pd.read_csv(file_path)                        required_cols = ["Total_Acceptances", "ForwardProb", "ReverseProb", "NumAtomsSwapped", "DeltaE"]            if not all(col in df.columns for col in required_cols):                continue                        fig, axes = plt.subplots(2, 2, figsize=(15, 10))            fig.suptitle(f'Move Analysis: {table_label}', fontsize=16, fontweight='bold')                        iterations = np.arange(len(df))                        axes[0, 0].plot(iterations, df["Total_Acceptances"], color=curve_color, linewidth=2)            axes[0, 0].set_title('Total Acceptances')            axes[0, 0].set_xlabel('Iteration')            axes[0, 0].set_ylabel('Acceptances')            axes[0, 0].grid(True, alpha=0.3)                        axes[0, 1].plot(iterations, df["NumAtomsSwapped"], color=curve_color, linewidth=2)            axes[0, 1].set_title('Atoms Swapped per Move')            axes[0, 1].set_xlabel('Iteration')            axes[0, 1].set_ylabel('Number of Atoms')            axes[0, 1].grid(True, alpha=0.3)                        axes[1, 0].plot(iterations, df["DeltaE"], color=curve_color, linewidth=2)            axes[1, 0].set_title('Energy Change per Move')            axes[1, 0].set_xlabel('Iteration')            axes[1, 0].set_ylabel('ΔE')            axes[1, 0].grid(True, alpha=0.3)                        prob_ratio = df["ForwardProb"] / df["ReverseProb"].replace(0, np.nan)            axes[1, 1].plot(iterations, prob_ratio, color=curve_color, linewidth=2)            axes[1, 1].set_title('Forward/Reverse Probability Ratio')            axes[1, 1].set_xlabel('Iteration')            axes[1, 1].set_ylabel('Forward/Reverse')            axes[1, 1].set_yscale('log')            axes[1, 1].grid(True, alpha=0.3)                        plt.tight_layout()            plt.show()        except Exception:            continuedef animation_driver(csv_files, table_labels, start_iteration=0, end_iteration=50, fps=10, dpi=72):    def preprocess_csv_to_npz(csv_file: Path, start_iteration: int, end_iteration: int) -> Optional[Path]:        npz_file_name = f"{csv_file.stem}_iter_{start_iteration}_to_{end_iteration}.npz"        npz_path = csv_file.parent / npz_file_name                try:            df_for_n = pd.read_csv(csv_file, dtype=str, usecols=["GridConfigStarting"], nrows=1)            if df_for_n.empty:                return None            config_str = df_for_n["GridConfigStarting"].iloc[0]            num_cells = len(str(config_str).strip('"\' ').split(';'))            N = int(math.sqrt(num_cells))            if N * N != num_cells:                return None                        nrows_to_read = end_iteration - start_iteration + 1            if nrows_to_read <= 0:                np.savez_compressed(npz_path, N=N, states=np.array([], dtype=np.int8).reshape(0, N, N))                return npz_path                        df_sim = pd.read_csv(csv_file, dtype=str, skiprows=range(1, start_iteration + 1), nrows=nrows_to_read)        except Exception:            return None                if df_sim.empty:            np.savez_compressed(npz_path, N=N, states=np.array([], dtype=np.int8).reshape(0, N, N))            return npz_path                all_states = [parse_grid_config_string_to_species_array(row["GridConfigStarting"], N) for _, row in df_sim.iterrows()]                if "GridConfigPotential" in df_sim.columns and not df_sim.empty:            final_potential = parse_grid_config_string_to_species_array(df_sim.iloc[-1]["GridConfigPotential"], N)            if final_potential is not None:                all_states.append(final_potential)                valid_states = [s for s in all_states if s is not None]                if len(valid_states) == 0:            np.savez_compressed(npz_path, N=N, states=np.array([], dtype=np.int8).reshape(0, N, N))            return npz_path                np.savez_compressed(npz_path, N=N, states=np.array(valid_states, dtype=np.int8))        return npz_path        def create_animation(npz_file: Path, output_filename: str, fps: int, dpi: int, start_iteration_offset: int):        try:            with np.load(npz_file) as data:                N = data['N']                states = data['states']        except Exception:            return                if states.size == 0 or len(states) <= 1:            return                total_frames = len(states) - 1        if total_frames <= 0:            return                    fig, ax = plt.subplots(figsize=(8, 8))        ax.set(xlim=(-0.5, N - 0.5), ylim=(-0.5, N - 0.5), aspect="equal", xticks=[], yticks=[])        fig.tight_layout()                cmap = ListedColormap(['#d62728', '#1f77b4'])        cell_rects = [ax.add_patch(plt.Rectangle((x - 0.5, y - 0.5), 1, 1)) for y in range(N) for x in range(N)]        status_text = ax.text(0.5, 1.02, '', transform=ax.transAxes, ha='center', fontsize=12)                initial_state_flat = states[0].flatten()        for i, rect in enumerate(cell_rects):            rect.set_facecolor(cmap(initial_state_flat[i]))            rect.set_edgecolor('none')                bordered_rects = set()                def update(frame_idx: int):            current_state = states[frame_idx]            next_state = states[frame_idx + 1]                        result_iter_num = start_iteration_offset + frame_idx + 1            status_text.set_text(f"Iteration {result_iter_num}")                        diff_indices = np.where(current_state.flatten() != next_state.flatten())[0]                        for rect_idx in bordered_rects:                cell_rects[rect_idx].set_linewidth(0)                cell_rects[rect_idx].set_edgecolor('none')            bordered_rects.clear()                        next_state_flat = next_state.flatten()            for idx in diff_indices:                rect = cell_rects[idx]                new_color = cmap(next_state_flat[idx])                rect.set_facecolor(new_color)                rect.set_edgecolor('yellow')                rect.set_linewidth(3)                bordered_rects.add(idx)                        return cell_rects + [status_text]                try:            Writer = animation.writers['ffmpeg']            writer = Writer(fps=fps, metadata=dict(artist='MC Simulation'), bitrate=-1,                           extra_args=['-preset', 'ultrafast', '-crf', '28'])            anim = animation.FuncAnimation(fig, update, frames=total_frames, blit=True, interval=1000 // fps)            anim.save(output_filename, writer=writer, dpi=dpi)                        if os.path.exists(output_filename):                display(Video(output_filename, embed=True, width=600, height=600))        except Exception:            pass        finally:            plt.close(fig)        for csv_file, table_label in zip(csv_files, table_labels):        csv_path = Path(csv_file)        if not csv_path.exists():            continue                npz_file_name = f"{csv_path.stem}_iter_{start_iteration}_to_{end_iteration}.npz"        npz_path = csv_path.parent / npz_file_name        output_file = Path(f"{csv_path.stem}_anim_iter_{start_iteration}_to_{end_iteration}.mp4")                generated_npz = preprocess_csv_to_npz(csv_path, start_iteration=start_iteration, end_iteration=end_iteration)        if not generated_npz:            continue                create_animation(generated_npz, str(output_file), fps, dpi, start_iteration)

In [ ]:
def simple_batch_processor(*filenames, run_energy=True, run_clusters=True, run_grids=True, run_moves=True, run_animations=False):    if not filenames:        return        csv_files = [f"{base_name}.csv" for base_name in filenames]    table_labels = list(filenames)        num_files = len(csv_files)    curve_colours = get_distinct_colors(num_files)        existing_files = [f for f in csv_files if Path(f).is_file()]    existing_labels = [table_labels[i] for i, f in enumerate(csv_files) if Path(f).is_file()]    existing_colours = [curve_colours[i] for i, f in enumerate(csv_files) if Path(f).is_file()]        if not existing_files:        return        if run_energy:        energy_analysis_driver(existing_files, existing_labels, existing_colours)        if run_clusters:        cluster_analysis_driver(existing_files, existing_labels, existing_colours)        if run_grids:        grid_visualization_driver(existing_files, existing_labels)        if run_moves:        moves_analysis_driver(existing_files, existing_labels, existing_colours)        if run_animations:        animation_driver(existing_files, existing_labels)def combinatoric_processor(*, BJ: str, include_kawasaki: bool = True,                         base_variants: List[str] = None, signifiers: List[List[str]] = None,                         cluster_types: List[str] = None, run_energy=True, run_clusters=True,                         run_grids=True, run_moves=True, run_animations=False):        base_variants = base_variants or ["DR"]    signifiers = signifiers or [["tol0.50"], ["scp0.50"]]    cluster_types = cluster_types or ["1", "2"]        csv_files, tbl, col, leg, leg_col, expand = build_filenames(        BJ=BJ, include_kawasaki=include_kawasaki,        base_variants=base_variants, signifiers=signifiers, cluster_types=cluster_types)        existing_files = [f for f in csv_files if Path(f).is_file()]    existing_labels = [tbl[i] for i, f in enumerate(csv_files) if Path(f).is_file()]    existing_colours = [col[i] for i, f in enumerate(csv_files) if Path(f).is_file()]        if not existing_files:        return        if run_energy:        energy_analysis_driver(existing_files, existing_labels, existing_colours, f"Combinatoric Analysis BJ={BJ}")        if run_clusters:        cluster_analysis_driver(existing_files, existing_labels, existing_colours)        if run_grids:        grid_visualization_driver(existing_files, existing_labels)        if run_moves:        moves_analysis_driver(existing_files, existing_labels, existing_colours)        if run_animations:        animation_driver(existing_files, existing_labels)

In [ ]:
USE_SIMPLE_BATCH = TrueUSE_COMBINATORIC = Falseif USE_SIMPLE_BATCH:    filenames_to_analyze = [        "BJ=-0.44_Kawasaki_0_tol0.00_scp0.50",        "BJ=-0.44_DR_1_tol0.50_scp0.50",        "BJ=-0.44_DR_2_tol0.50_scp0.50"    ]        simple_batch_processor(        *filenames_to_analyze,        run_energy=True,        run_clusters=True,        run_grids=True,        run_moves=True,        run_animations=False    )if USE_COMBINATORIC:    combinatoric_processor(        BJ="-0.44",        include_kawasaki=True,        base_variants=["DR", "NewMC"],        signifiers=[["tol0.50", "tol0.25"], ["scp0.50", "scp0.25"]],        cluster_types=["1", "2"],        run_energy=True,        run_clusters=True,        run_grids=False,        run_moves=True,        run_animations=False    )